# Part 1: The question and the data

Final Predictive question: How well does lead time predict booking cancellations, and does its predictive value vary across market segments?

To recap the ending of our 01 findings, our original predictive question included lead time, country, and market segment. During the data exploration phase, we found out that Portugal was an extreme outlier. After the feedback from the TA, we were pushed to find that Portugal was actually the default label until check in. This revelation forced us to drop country as a prediction variable (for extended reasoning, refer to 01).  

We as a team believe that this predictive question will greatly benefit a revenue manager who is responsible for managing overbookings in hotel. As you may know, a common practice in the industry to maximize profit is to overbook rooms in advance in the anticipation that a certain percentage of those rooms will be cancelled. Creating a powerful model that accurately predict whether a booking will cancel can help our revenue manager maximize this opportunity without going overboard and creating a potential loss.

By assigning the potential gain/losses with a concrete gain/loss for the business, we can further tune our model to not only minimize error but to also maximize profit.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-02-11/hotels.csv"
df = pd.read_csv(url)
print('bookings:', len(df), '| cancellation rate: %.1f%%' % (df['is_canceled'].mean() * 100))

bookings: 119390 | cancellation rate: 37.0%


In [2]:
# --- Lucas: Non Refund check ---
non_refund = df[df['deposit_type'] == 'Non Refund']

# Agents with Non Refund bookings, and how many cancelled every one of them
by_agent = non_refund.dropna(subset=['agent']).groupby('agent')['is_canceled'].mean()
print(f'{(by_agent == 1).sum()} of {len(by_agent)} agents cancelled every Non Refund booking')

# Cancellation rate by the quarter the booking was made (arrival date minus lead time)
arrival = pd.to_datetime(df['arrival_date_year'].astype(str) + '-' + df['arrival_date_month'] + '-'
                         + df['arrival_date_day_of_month'].astype(str), format='%Y-%B-%d')
booked = arrival - pd.to_timedelta(df['lead_time'], unit='D')
quarter = booked[non_refund.index].dt.to_period('Q').rename('booked in')
by_quarter = non_refund.groupby(quarter)['is_canceled'].agg(bookings='size', cancel_rate='mean')
by_quarter['cancel_rate'] = (by_quarter['cancel_rate'] * 100).round(1)
print('\nNon Refund bookings by booking quarter:')
print(by_quarter)

81 of 87 agents cancelled every Non Refund booking

Non Refund bookings by booking quarter:
           bookings  cancel_rate
booked in                       
2014Q4         1562        100.0
2015Q1          761        100.0
2015Q2          482        100.0
2015Q3         2963        100.0
2015Q4         2766         98.2
2016Q1         1758         98.4
2016Q2          709         99.2
2016Q3          227        100.0
2016Q4         2060        100.0
2017Q1         1213         99.4
2017Q2           83        100.0
2017Q3            3         66.7


In [3]:
# --- Lucas: which rows and columns the model can use ---
# Non Refund rows leave the model because they almost always cancel, so there is nothing to learn.
# Part 5 shows the effect of keeping them.
data = df[df['deposit_type'] != 'Non Refund'].copy()

TARGET = 'is_canceled'
LEAKAGE = ['reservation_status',           # this IS the outcome
           'reservation_status_date',      # the date that status was set
           'assigned_room_type',           # given at check-in
           'booking_changes',              # counted up to check-in or cancellation
           'country',                      # default Portugal until check-in, see 01
           'required_car_parking_spaces']  # no booking with a space ever cancels, printed below
IDS = ['agent', 'company']                 # labels for the booking, not information about it
OTHER = ['arrival_date_year',              # a future booking's year never appears in training
         'previous_cancellations']         # changes meaning over time, printed below

features = [c for c in data.columns if c not in LEAKAGE + IDS + OTHER + [TARGET]]
cat_cols = data[features].select_dtypes('object').columns.tolist()
num_cols = [c for c in features if c not in cat_cols]

# Evidence for the last two decisions above
parked = data[data['required_car_parking_spaces'] > 0]
print(f"bookings with a parking space: {len(parked)} | cancelled: {parked[TARGET].sum()}")
repeat = data[data['previous_cancellations'] > 0].groupby('arrival_date_year')[TARGET].mean() * 100
print('\nCancellation rate (%) of bookings with a previous cancellation, by arrival year:')
print(repeat.round(1).to_string())

print('\nbookings left:', len(data), '| cancellation rate: %.1f%%' % (data[TARGET].mean() * 100))
print('highest adr left:', data['adr'].max())
print('\nInputs:')
print(pd.DataFrame({'type': ['numeric' if c in num_cols else 'categorical' for c in features],
                    'missing': data[features].isna().sum().values}, index=features).to_string())

bookings with a parking space: 7415 | cancelled: 0

Cancellation rate (%) of bookings with a previous cancellation, by arrival year:
arrival_date_year
2015    97.6
2016    58.4
2017    17.3

bookings left: 104803 | cancellation rate: 28.4%
highest adr left: 510.0

Inputs:
                                       type  missing
hotel                           categorical        0
lead_time                           numeric        0
arrival_date_month              categorical        0
arrival_date_week_number            numeric        0
arrival_date_day_of_month           numeric        0
stays_in_weekend_nights             numeric        0
stays_in_week_nights                numeric        0
adults                              numeric        0
children                            numeric        4
babies                              numeric        0
meal                            categorical        0
market_segment                  categorical        0
distribution_channel            catego

From our Part 1, you can see that 81 of 87 agents cancelled every Non Refund booking and the rate is 98% -100% in nearly every booking quarter. Removing these bookings will leave us with 104,803 bookings with a 28.4% cancellation rate.

Below is a summary of leaked columns that is recorded at or after check-in or cancellation, so the hotel wouldn't know it when making the prediction.

| `reservation_status`, `reservation_status_date` | These are the outcome itself, and the date it was set |
| `assigned_room_type`, `booking_changes` | Set at check-in, or counted up to check-in or cancellation |
| `country` | Default Portugal until check-in (see `01`) |
| `required_car_parking_spaces` | 7,415 bookings had a parking space, and 0 cancelled, which means it's recorded at arrival |
| `agent`, `company` | Labels for the booking, not information about it |
| `arrival_date_year` | A future booking's year never appears in training |
| `previous_cancellations` | Bookings with a previous cancellation cancelled at 97.6% in 2015, 58.4% in 2016 and 17.3% in 2017 |
| What's left | 21 inputs: 13 numeric and 8 categorical. Only `children` has missing values (4), filled with the median |
| Assumption | `total_of_special_requests` is kept, though you can't confirm it's known at booking |